# Hari 25 — Evaluation Ronde Kedua (Final)

**Ini keputusan terakhir.** Tidak ada iterasi ketiga — hasil apa pun yang keluar hari ini, kita dokumentasikan dan lanjut ke Deployment. Metodologinya sama seperti Hari 22 ("Uji Kewajaran": perbandingan fold-demi-fold yang adil terhadap baseline), tapi sekarang pakai `dataset_siap_modeling_v2.csv` (18 fitur, termasuk `lag_0`).

**Benchmark CV lama (v1, dari Hari 19-21) yang harus dibandingkan:**
- Baseline (per fold, dari Hari 22): rata-rata 15.226,82
- Linear Regression: rata-rata 73.013,00 (std 105.471,04) — 2 dari 5 fold menang
- Random Forest tuned: rata-rata 53.164,83 (std 36.824,76) — 0 dari 5 fold menang

In [17]:
# Cell ini sudah lengkap. Catatan: kita tidak perlu lagi reload dataset_bersih_minggu2.csv
# untuk current_actual seperti Hari 22 — kolom lag_0 sekarang PERSIS itu, jadi lebih ringkas.

import pandas as pd
#import numpy as np
import joblib
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

df_fitur_v2 = pd.read_csv("dataset_siap_modeling_v2.csv", index_col=0, parse_dates=True)
X_full_v2 = df_fitur_v2.drop(columns=["target_minggu_depan"])
y_full_v2 = df_fitur_v2["target_minggu_depan"]
current_actual_full_v2 = X_full_v2["lag_0"]

print(f"Data v2: {X_full_v2.shape[0]} baris, {X_full_v2.shape[1]} fitur")

Data v2: 143 baris, 18 fitur


## Perbandingan Adil Fold-demi-Fold (v2)

In [18]:
# Cell ini sudah lengkap — sama persis metodologinya dengan Hari 22.

tscv = TimeSeriesSplit(n_splits=5)
baseline_scores, linear_scores, rf_scores = [], [], []

for train_idx, test_idx in tscv.split(X_full_v2):
    y_test_fold = y_full_v2.iloc[test_idx]
    current_fold = current_actual_full_v2.iloc[test_idx]
    baseline_scores.append(mean_absolute_error(y_test_fold, current_fold))

    X_train_fold, X_test_fold = X_full_v2.iloc[train_idx], X_full_v2.iloc[test_idx]
    y_train_fold = y_full_v2.iloc[train_idx]

    scaler_fold = StandardScaler().fit(X_train_fold)
    lin = LinearRegression().fit(scaler_fold.transform(X_train_fold), y_train_fold)
    linear_scores.append(mean_absolute_error(y_test_fold, lin.predict(scaler_fold.transform(X_test_fold))))

    rf = RandomForestRegressor(random_state=42, max_depth=7, min_samples_leaf=2).fit(X_train_fold, y_train_fold)
    rf_scores.append(mean_absolute_error(y_test_fold, rf.predict(X_test_fold)))

perbandingan_v2 = pd.DataFrame({
    "Baseline (per fold)": baseline_scores,
    "Linear Regression": linear_scores,
    "Random Forest (tuned)": rf_scores,
})
print(perbandingan_v2)
print("\nRata-rata & Std:")
print(perbandingan_v2.agg(["mean", "std"]))

   Baseline (per fold)  Linear Regression  Random Forest (tuned)
0          5503.478261       14714.994072           21090.140286
1         26558.260870       40801.815651           65429.295439
2         11532.173913      282551.498499           43789.141835
3         27308.130435       23350.215870           31139.826901
4          5232.043478        3646.495843            5903.039096

Rata-rata & Std:
      Baseline (per fold)  Linear Regression  Random Forest (tuned)
mean         15226.817391       73013.003987           33470.288711
std          10982.360357      117920.210600           22619.180022


## Latihan: Hitung Konsistensi Menang & Bandingkan ke v1

In [19]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat kolom "Linear Menang" dan "RF Menang" di perbandingan_v2, sama seperti Hari 22
#    (bandingkan MAE model < MAE baseline pada fold yang sama)
# 2. Cetak perbandingan_v2 dengan kolom tambahan itu
# 3. Hitung total fold menang untuk masing-masing model
# 4. Cetak perbandingan RINGKAS v1 vs v2 (salin manual angka v1 dari markdown di atas):
#    - Linear Regression: v1 (mean 73013.00, 2/5 menang) vs v2 (mean & jumlah menang kamu)
#    - Random Forest tuned: v1 (mean 53164.83, 0/5 menang) vs v2 (mean & jumlah menang kamu)
# Tulis kode kamu di bawah ini:
perbandingan_v2["Linear Menang"] = (
    perbandingan_v2["Linear Regression"] < perbandingan_v2["Baseline (per fold)"])
perbandingan_v2["RF Menang"] = (
    perbandingan_v2["Random Forest (tuned)"] < perbandingan_v2["Baseline (per fold)"])
print(perbandingan_v2)

print(f"Linear menang di {perbandingan_v2['Linear Menang'].sum()} dari {len(perbandingan_v2)} fold")
print(f"Random Forest menang di {perbandingan_v2['RF Menang'].sum()} dari {len(perbandingan_v2)} fold")

print("Perbandingan v1 vs v2")
print("Linear Regression v1 = 2/5 menang, v2 = "
      f"{perbandingan_v2['Linear Menang'].sum()}/{len(perbandingan_v2)} menang")
print("Random Forest (tuned) v1 = 0/5 menang, v2 = "
      f"{perbandingan_v2['RF Menang'].sum()}/{len(perbandingan_v2)} menang")

   Baseline (per fold)  Linear Regression  Random Forest (tuned)  \
0          5503.478261       14714.994072           21090.140286   
1         26558.260870       40801.815651           65429.295439   
2         11532.173913      282551.498499           43789.141835   
3         27308.130435       23350.215870           31139.826901   
4          5232.043478        3646.495843            5903.039096   

   Linear Menang  RF Menang  
0          False      False  
1          False      False  
2          False      False  
3           True      False  
4           True      False  
Linear menang di 2 dari 5 fold
Random Forest menang di 0 dari 5 fold
Perbandingan v1 vs v2
Linear Regression v1 = 2/5 menang, v2 = 2/5 menang
Random Forest (tuned) v1 = 0/5 menang, v2 = 0/5 menang


## Tabel Ringkasan Lengkap: v1 vs v2, Semua Metrik

Lengkapi tabel ini secara manual dengan angka-angka yang sudah terkumpul dari Hari 21, 22, 24, dan hasil di atas:

| Model | MAE Single-split (v1 → v2) | Trend Accuracy (v1 → v2) | CV Mean (v1 → v2) | Fold Menang Baseline (v1 → v2) |
|---|---|---|---|---|
| Linear Regression | 7.806,56 → 7.806,56 | 62,07% → 62,07% | 73.013,00 → 73.013,00 | 2/5 → 2/5 |
| Random Forest (tuned) | 8.456,32 → 5.913,70 | 31,03% → 48,28% | 53.164,83 → 33470.28 | 0/5 → 0/5 |

## Checklist Final: Project Charter Hari 2

| Kriteria | Terpenuhi? | Catatan |
|---|---|---|
| Technical: MAE model < baseline (single-split) | ✅ / ❌ | ✅ |
| Technical: MAE model < baseline (konsisten di CV, tiap fold) | ✅ / ❌ / Sebagian | Sebagian |
| Technical: perbaikan minimal 15-20% dari baseline | ✅ / ❌ | ✅ |
| Business: output bisa diringkas jadi label Naik/Turun/Stabil | ✅ / ❌ | ✅ |
| Business: masyarakat awam bisa paham tanpa latar belakang statistik | ✅ / ❌ / Belum diuji ke pengguna asli | Belum diuji ke pengguna asli |

## Keputusan Go/No-Go FINAL

> **Keputusan:** Lanjut ke Deployment (wajib kali ini — tidak ada iterasi ketiga) → **Lanjut**
>
> **Model yang dipilih:** Linear Regression v2 / Random Forest v2 (coret salah satu) → **Linear Regression v2**
>
> **Alasan:** Random forest (tuned) tetap tidak menang fold apapun walaupun sudah di tambah fitur lag_0
>
> **Keterbatasan permanen yang didokumentasikan** (bukan lagi "akan diperbaiki nanti", tapi diterima sebagai bagian dari batasan proyek final):
> - **Tidak ada data vaksinasi sebelum Januari 2021 (diisi 0)**
> - **belum pernah diuji ke pengguna sungguhan**
>
> **Disclaimer untuk pengguna aplikasi (dipakai langsung di Streamlit nanti):** → performa model kurang stabil di periode-periode tertentu

## Simpan Model Final v2

In [20]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Muat X_train_v2.csv, X_train_scaled_v2.csv, y_train_v2.csv (kalau belum ada di memori)
# 2. Latih ulang model PILIHANMU dari Keputusan Go/No-Go di atas pada seluruh data
#    training v2 (X_train_v2 kalau Random Forest, X_train_scaled_v2 kalau Linear Regression)
# 3. Simpan sebagai model_final_v2 = model yang baru dilatih
# 4. joblib.dump(model_final_v2, "model_final_v2.pkl")
# 5. Cetak konfirmasi: nama model, apakah butuh data ter-scale, dan daftar 18 nama
#    kolom fitur yang dibutuhkan model ini (X_train_v2.columns.tolist()) — supaya
#    di Hari 26-27 tidak salah susun input untuk Streamlit
# Tulis kode kamu di bawah ini:
X_train_v2 = pd.read_csv("X_train_v2.csv", index_col=0, parse_dates=True)
X_train_scaled_v2 = pd.read_csv("X_train_scaled_v2.csv", index_col=0, parse_dates=True)
y_train_v2 = pd.read_csv("y_train_v2.csv", index_col=0, parse_dates=True).iloc[:, 0]

model_linear_v2 = LinearRegression().fit(X_train_scaled_v2, y_train_v2)
model_final_v2 = model_linear_v2
joblib.dump(model_final_v2, "model_final_v2.pkl")
print("Model final yang disimpan: Linear Regression v2")
print(X_train_v2.columns.tolist())

Model final yang disimpan: Linear Regression v2
['lag_1', 'lag_2', 'lag_3', 'rolling_mean_4w', 'vaksin_persen', 'bulan_1', 'bulan_2', 'bulan_3', 'bulan_4', 'bulan_5', 'bulan_6', 'bulan_7', 'bulan_8', 'bulan_9', 'bulan_10', 'bulan_11', 'bulan_12', 'lag_0']


## Modeling & Evaluation Summary Report — Perjalanan Lengkap

**Proyek:** Prediksi Tren Kasus COVID-19 Mingguan Indonesia

**Ringkasan perjalanan iterasi (Hari 15-25):**
1. Baseline naive forecast ditetapkan (Hari 2): MAE 12.953
2. Linear Regression & Random Forest dilatih (Hari 16-17), tuning (Hari 19-20)
3. Cross-validation `TimeSeriesSplit` mengungkap ketidakstabilan yang tersembunyi di balik hasil single-split (Hari 19-21)
4. **Iterasi 1 (Evaluation, Hari 22):** ditemukan baseline yang dipakai sebagai pembanding tidak adil secara metodologis — setelah dikoreksi (dihitung per fold), model masih kalah konsisten
5. **Keputusan putar balik** ke Data Preparation: menambah fitur `lag_0` yang sebelumnya terlewat (Hari 23)
6. **Iterasi 2 (Modeling, Hari 24):** retrain dengan `lag_0` — Random Forest terbukti sangat terbantu (MAE 8.456 → 5.914), Linear Regression relatif tidak berubah karena multikolinearitas parah
7. **Evaluation final (Hari 25):** (isi kesimpulan akhir kamu)

**Model final:** (isi dari Keputusan Go/No-Go di atas)

**Pelajaran metodologis terbesar dari seluruh Minggu 3-4:** metrik tunggal dari satu train-test split bisa sangat menyesatkan untuk data time-series kecil — cross-validation yang benar, perbandingan baseline yang adil, dan kesediaan untuk **putar balik ke fase sebelumnya** (bukan cuma maju terus) adalah bagian nyata dari proses data science, bukan tanda kegagalan.

## Refleksi Hari 25

> 1. Apakah menambah `lag_0` berhasil membuat model (yang kamu pilih) menang lebih konsisten di cross-validation dibanding versi lama? → **Random Forest tidak menang fold apa pun di keduanya, tapi rata-rata error dan variabilitasnya membaik**
> 2. Dari seluruh proses iterasi Hari 22-25 ini, menurutmu apa nilai paling penting yang didapat — bukan dari sisi angka, tapi dari sisi PROSES kerja data science? → **Memeperbaiki atau menambah fitur yang dibutuhkan**
> 3. Kalau kamu harus jelaskan ke seseorang yang tidak teknis kenapa proyek ini butuh "muter balik" segala, bagaimana kamu akan menjelaskannya dengan bahasa sederhana? → **kita harus menguji bebarapa model dan fitur, jika kurang bagus kita harus putar balik**

---
### Selanjutnya: Hari 26-30

- **Hari 26**: Interpretasi final model_final_v2, tulis dokumentasi batasan model untuk pengguna
- **Hari 27-28**: Bangun aplikasi Streamlit — input data terbaru, output prediksi + label tren + disclaimer
- **Hari 29-30**: Dokumentasi penutup 30 hari — README lengkap, refleksi perjalanan, siap jadi portofolio